In [2]:
import pandas as pd

In [3]:
# Load data
data = pd.read_csv("../data/raw/world_cities.csv")
data.head()

,country,name,lat,lng
0,AD,les Escaldes,42.50729,1.53414
1,AD,Andorra la Vella,42.50779,1.52109
2,AE,Warīsān,25.16744,55.40708
3,AE,Umm Suqaym,25.15491,55.21015
4,AE,Umm Al Quwain City,25.56473,55.55517


In [4]:
data.shape

(34135, 4)

In [5]:
# Filter countries for Europe and North America only
europe = ["AL","AD","AT","BY","BE","BA","BG","HR","CY","CZ","DK","EE","FI","FR",
          "DE","GR","HU","IS","IE","IT","LV","LI","LT","LU","MT","MD","MC","ME",
          "NL","MK","NO","PL","PT","RO","RS","SK","SI","ES","SE","CH","UA","GB"]

north_america = ["US","CA","MX"]

target_countries = europe + north_america
target_cities = data[data["country"].isin(target_countries)]

In [6]:
# Random sample with sample size = 300
cities = target_cities.sample(n=300, random_state=42)
cities.shape

(300, 4)

In [7]:
# Randomly assign growth potential score to each city
import numpy as np
np.random.seed(42)

cities["growth_potential"]=np.random.normal(loc=50, scale=15, size=len(cities))
cities[["name","growth_potential"]].head()

,name,growth_potential
23735,Grimstad,57.450712
31176,Lexington,47.926035
8248,Weißensee,59.715328
14064,Dún Laoghaire,72.845448
31098,Huntington,46.487699


In [8]:
# Create test period in months (datetimeindex)
months = pd.date_range(start="2025-01-01", end="2026-03-01", freq="MS")
# Months Dataframe
months_df = pd.DataFrame({"month":months})
months_df.head()

,month
0,2025-01-01
1,2025-02-01
2,2025-03-01
3,2025-04-01
4,2025-05-01


In [9]:
# Assign rollout months by growth potential (6 groups)

cities["rollout_wave"] = pd.qcut(cities["growth_potential"], q=6, labels=[5,4,3,2,1,0])


In [10]:
cities[["name","growth_potential","rollout_wave"]].sort_values("growth_potential",ascending=False).head(10)

,name,growth_potential,rollout_wave
11668,Châlons-en-Champagne,107.790972,0
31015,Niles,90.802537,0
29375,Vidradnyi,86.948632,0
1564,Sevlievo,84.719879,0
32188,Gardena,82.856834,0
25039,Lubartów,82.159161,0
4458,Saint John West,81.995501,0
10193,Maó,81.832343,0
29064,Volochysk,78.451895,0
30997,Maywood,78.292789,0


In [11]:
# Map rollout wave to actual month using dictionary 

rollout_months = {
    0: pd.Timestamp("2025-07-01"),
    1: pd.Timestamp("2025-08-01"),
    2: pd.Timestamp("2025-09-01"),
    3: pd.Timestamp("2025-10-01"),
    4: pd.Timestamp("2025-11-01"),
    5: pd.Timestamp("2025-12-01"),
}

cities["go_live_date"] = cities["rollout_wave"].map(rollout_months)
cities[["name", "growth_potential", "rollout_wave", "go_live_date"]].head(10)

,name,growth_potential,rollout_wave,go_live_date
23735,Grimstad,57.450712,1,2025-08-01
31176,Lexington,47.926035,3,2025-10-01
8248,Weißensee,59.715328,1,2025-08-01
14064,Dún Laoghaire,72.845448,0,2025-07-01
31098,Huntington,46.487699,3,2025-10-01
32174,Fair Oaks,46.487946,3,2025-10-01
31423,Dover,73.688192,0,2025-07-01
32278,Monterey,61.511521,1,2025-08-01
30246,Webster Groves,42.957884,4,2025-11-01
4245,Brossard,58.138401,1,2025-08-01


In [12]:
# Cross join to create panel
panel = cities.merge(months_df, how="cross")
panel.shape
panel.head()

,country,name,lat,lng,growth_potential,rollout_wave,go_live_date,month
0,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-01-01
1,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-02-01
2,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-03-01
3,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-04-01
4,NO,Grimstad,58.3405,8.59343,57.450712,1,2025-08-01,2025-05-01


In [13]:
# Add flag for treated groups - FALSE before go live date

panel["treated"] = panel["month"] >= panel["go_live_date"]
panel[["name","month","go_live_date","treated"]].sample(10)

,name,month,go_live_date,treated
414,Moose Jaw,2025-10-01,2025-08-01,True
3388,Karcag,2026-02-01,2025-07-01,True
144,Brossard,2025-10-01,2025-08-01,True
1183,Ottawa,2026-02-01,2025-09-01,True
3075,Rákospalota,2025-01-01,2025-11-01,False
387,Cornwall,2026-01-01,2025-09-01,True
4191,Steyr,2025-07-01,2025-10-01,False
2586,Pradl,2025-07-01,2025-10-01,False
2743,Zacapu,2026-02-01,2025-12-01,True
1024,Tavares,2025-05-01,2025-09-01,False


In [14]:
# Set retention rates based on growth potential and base retention rate
base_retention = 0.25
growth_effect = (panel["growth_potential"]-50)*0.002
panel["baseline_outcome"]=base_retention+growth_effect
panel[["name", "growth_potential", "baseline_outcome"]].sort_values("growth_potential", ascending=False).head(5)

,name,growth_potential,baseline_outcome
3142,Châlons-en-Champagne,107.790972,0.365582
3143,Châlons-en-Champagne,107.790972,0.365582
3136,Châlons-en-Champagne,107.790972,0.365582
3137,Châlons-en-Champagne,107.790972,0.365582
3138,Châlons-en-Champagne,107.790972,0.365582


In [15]:
# Introduce Seasonal Impact

seasonal_effect_by_month = {
    1: 0.02,   # January resurgence
    2: 0.00,
    3: 0.00,
    4: 0.00,
    5: -0.01,
    6: -0.03,  # summer lull begins
    7: -0.04,  # summer lull
    8: -0.03,  # summer lull / back-to-school prep
    9: 0.03,   # back-to-school bump
    10: 0.01,
    11: 0.00,
    12: -0.01, # holidays
}

panel["month_num"]=panel["month"].dt.month
panel["seasonal_effect"]=panel["month_num"].map(seasonal_effect_by_month)
panel["outcome_no_treatment"]= panel["baseline_outcome"]+ panel["seasonal_effect"]
panel[["name", "month", "baseline_outcome", "seasonal_effect", "outcome_no_treatment"]].sample(8)

,name,month,baseline_outcome,seasonal_effect,outcome_no_treatment
1087,Rinkeby,2025-08-01,0.248925,-0.03,0.218925
695,Waltham,2025-06-01,0.236181,-0.03,0.206181
1431,Békéscsaba,2025-07-01,0.206095,-0.04,0.166095
2612,Gateshead,2025-03-01,0.258301,0.00,0.258301
4460,Maspalomas,2025-06-01,0.259219,-0.03,0.229219
3283,Minden,2026-02-01,0.235439,0.00,0.235439
3021,Guadalupe,2025-07-01,0.266824,-0.04,0.226824
1536,Santutxu,2025-07-01,0.239719,-0.04,0.199719


In [19]:
# Add the treatment effect 
true_effect = 0.04 # +0.04 percentage points

panel["outcome"] = panel["outcome_no_treatment"] + (panel["treated"]*true_effect)
panel[["name", "month", "treated", "outcome_no_treatment", "outcome"]].sample(8)

,name,month,treated,outcome_no_treatment,outcome
3812,Blanquefort,2025-03-01,False,0.204419,0.204419
4180,Isernia,2025-11-01,True,0.252094,0.292094
3006,Bethany,2025-07-01,False,0.220734,0.220734
4069,Nogent-sur-Oise,2025-05-01,False,0.196924,0.196924
589,Oleksandriya,2025-05-01,False,0.245906,0.245906
857,Santa Eulària des Riu,2025-03-01,False,0.240724,0.240724
4166,Collinwood,2025-12-01,True,0.221993,0.261993
680,Izegem,2025-06-01,False,0.198405,0.198405
